# Stage 0: which Qwen3.5-4B layer makes extraction readable?

The backbone is frozen. One span+null probe head per layer is trained on the same batches, then every head
predicts on held-out sets. The question: can a frozen base already locate the answer span and detect absent
fields, and at which depth? This decides whether we move on to heads-only training and then LoRA.

Scope: single-span field types (`text`, `number`, `date`, `time`, `datetime`, `duration`) and null detection.
Compute fields only train the null head; list, estimate and forecast fields are left out (stage 1+).

**Run:** Runtime → Change runtime type → A100. Set `REPO_URL` below, then Run all.
`DRY_RUN=True` runs everything on CPU with a tiny random backbone and small data (a smoke test).

In [ ]:
# --- config ---
REPO_URL = "https://github.com/pherber3/extraction-model.git"
BRANCH = "main"
MODEL_ID = "Qwen/Qwen3.5-4B"
DRY_RUN = False

N_SYNTH = 3000          # synthetic docs (train pool; one template per doc type held out for eval)
N_SQUAD = 3000          # SQuAD 2.0 train paragraphs (multi-question records)
N_GRETEL = 2000         # Gretel synthetic finance docs (train)
N_EVAL = 400            # docs per eval suite
EPOCHS = 2
LAYERS = "all"          # or a list of 1-based layer indices
OUT_DIR = "/content/stage0_results"
USE_DRIVE = False       # True: also copy results to Drive

if DRY_RUN:
    N_SYNTH, N_SQUAD, N_GRETEL, N_EVAL, EPOCHS = 120, 60, 40, 30, 1

In [ ]:
# --- setup: clone/pull the repo, install what Colab lacks ---
import os, subprocess, sys
REPO_DIR = "/content/extraction-model"
if os.path.isdir("/content"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    pkgs = ["transformers>=5.20", "datasets"] + ([] if DRY_RUN else ["flash-linear-attention"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
else:  # running locally from a repo checkout
    REPO_DIR = os.path.abspath("..")
    OUT_DIR = os.path.join(REPO_DIR, "results", "stage0")
sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "cuda", torch.cuda.is_available())

## Data

In [ ]:
import collections, json, random
from extraction import synth, public_data

synth_all = synth.generate(N_SYNTH, seed=0)
templates = sorted({ex["template_id"] for ex in synth_all})
by_type = collections.defaultdict(list)
for t in templates:
    by_type[t.split("/")[0]].append(t)
holdout = {random.Random(0).choice(ts) for ts in by_type.values()}   # one unseen template per doc type
synth_train, synth_eval = synth.split_by_template(synth_all, holdout)

train = synth_train + public_data.squad_v2("train", N_SQUAD) + public_data.gretel_finance("train", N_GRETEL)
eval_suites = {
    "synth_unseen_templates": synth_eval[:N_EVAL],
    "squad_v2_dev": public_data.squad_v2("validation", N_EVAL, seed=1),
    "gretel_test": public_data.gretel_finance("test", N_EVAL, seed=1),
    "fda": public_data.fda(N_EVAL),
}
def describe(examples):
    kinds = collections.Counter(g["kind"] for ex in examples for g in ex["gold"].values())
    return f"{len(examples)} docs, {sum(kinds.values())} fields {dict(kinds)}"
print("holdout templates:", sorted(holdout))
print("train:", describe(train))
for name, ex in eval_suites.items():
    print(f"{name}:", describe(ex))

## Backbone

In [ ]:
from extraction import probe
if DRY_RUN:
    backbone = probe.tiny_backbone(MODEL_ID)
else:
    backbone = probe.load_backbone(MODEL_ID, device="cuda", dtype=torch.bfloat16)
    print(f"GPU memory after load: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
layers = list(range(1, backbone.num_layers + 1)) if LAYERS == "all" else list(LAYERS)
full_attention = {i + 1 for i, t in enumerate(backbone.layer_types) if t == "full_attention"}
print(f"{backbone.num_layers} layers; full attention after layers {sorted(full_attention)}")
config = probe.ProbeConfig(layers=layers, epochs=EPOCHS, max_tokens=4096 if DRY_RUN else 16384)

## Train one probe per layer

In [ ]:
import time
t0 = time.perf_counter()
result = probe.train_probes(backbone, train, config)
print(f"training: {time.perf_counter() - t0:.0f}s")
probe.save_heads(result, os.path.join(OUT_DIR, "heads"))

## Evaluate every layer on every suite

Accuracy counts null as an answer (correct when the field is absent and the model says null).
`extract_acc` is accuracy on fields whose gold is a span; `null_brier` scores `p_null` against absence.

In [ ]:
import numpy as np
from extraction import scoring
from extraction.encode import SPAN_TYPES

def span_rows(rows):
    return [r for r in rows if r["type"] in SPAN_TYPES and r["gold_kind"] != "compute"]

table, all_rows = [], {}
for suite, examples in eval_suites.items():
    t0 = time.perf_counter()
    preds = probe.predict(backbone, result.heads, examples, config)
    for layer in layers:
        rows = span_rows(scoring.score_predictions(examples, preds[layer]))
        all_rows[(suite, layer)] = rows
        correct = np.array([r["correct"] for r in rows], dtype=float)
        is_null = np.array([r["gold_kind"] == "null" for r in rows])
        p_null = np.array([np.nan if r["p_null"] is None else r["p_null"] for r in rows], dtype=float)
        answered = ~np.isnan(p_null)  # unanswered = state truncated away; counted wrong in acc
        table.append({
            "suite": suite, "layer": layer, "attn": "full" if layer in full_attention else "linear",
            "n": len(rows), "acc": correct.mean(),
            "extract_acc": correct[~is_null].mean() if (~is_null).any() else float("nan"),
            "null_acc": correct[is_null].mean() if is_null.any() else float("nan"),
            "null_brier": scoring.brier(p_null[answered], is_null[answered].astype(float)),
        })
    print(f"{suite}: {time.perf_counter() - t0:.0f}s")

with open(os.path.join(OUT_DIR, "layer_table.json"), "w") as f:
    json.dump(table, f, indent=1)
for suite in eval_suites:
    print(f"\n== {suite}")
    print(f"{'layer':>5} {'attn':>6} {'n':>5} {'acc':>6} {'extract':>8} {'null':>6} {'brier':>6}")
    for r in (r for r in table if r["suite"] == suite):
        print(f"{r['layer']:>5} {r['attn']:>6} {r['n']:>5} {r['acc']:6.3f} {r['extract_acc']:8.3f} {r['null_acc']:6.3f} {r['null_brier']:6.3f}")

## Best layer: per-type breakdown and sample errors

In [ ]:
def mean_acc(layer):
    return np.mean([r["acc"] for r in table if r["layer"] == layer])
best = max(layers, key=mean_acc)
print(f"best layer by mean accuracy across suites: {best} ({'full' if best in full_attention else 'linear'} attention)")
for suite in eval_suites:
    rows = all_rows[(suite, best)]
    summary = scoring.summarize(rows)
    print(f"\n== {suite} @ layer {best}")
    print(json.dumps({k: summary[k] for k in summary if k in ("accuracy", "by_type", "by_gold_kind", "null_detection", "calibration")}, indent=1, default=float))
    errors = [r for r in rows if not r["correct"]][:5]
    for r in errors:
        print(f"  {r['id']} {r['field']}: gold={r['gold_value']!r} pred={r['pred_value']!r} p_null={r['p_null']:.2f}")
with open(os.path.join(OUT_DIR, f"rows_layer{best}.json"), "w") as f:
    json.dump({suite: all_rows[(suite, best)] for suite in eval_suites}, f, default=str)

## Save results

In [ ]:
import shutil
if USE_DRIVE and not DRY_RUN:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(OUT_DIR, "/content/drive/MyDrive/stage0_results", dirs_exist_ok=True)
archive = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("wrote", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass